In [13]:
import os 
import glob
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 600
import matplotlib.path as mpath
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Polygon
import math
import matplotlib.colors as mcolors
import numpy.ma as ma
import random
import pandas as pd
import iris
import iris.coords as icoords
import iris.cube as icube
import iris.coord_categorisation
import datetime
from collections import Counter
import logging
from matplotlib.patches import Patch

# Set logging level to WARNING to avoid debug logs
logging.getLogger('matplotlib.font_manager').setLevel(logging.WARNING)
import iris.analysis.cartography as icart
import iris.plot as iplt
#import iris.quickplot as iplt
import cartopy
import cartopy.crs as ccrs
import iris.analysis.cartography
from copy import deepcopy
import warnings
import matplotlib as mlt
import matplotlib.pyplot as plt
from iris.time import PartialDateTime
plt.rcParams.update({'font.size': 16})
from scipy.stats import percentileofscore
import seaborn as sns
from iris.analysis import Linear
from scipy.spatial import ConvexHull

from scipy.ndimage import gaussian_filter
from sklearn.covariance import EllipticEnvelope
from numpy import nan


# functions

In [2]:
def is_jja(date):
    ''' check if date is in JJA, for blocks'''
    return str(date)[4:6] in ['06', '07', '08']

In [3]:
def adjust_date(date, offset = 15):
    year = int(date[:4])
    month = int(date[4:6])
    day = int(date[6:])

    day += offset
    if day > 30:
        day = day -30
        month += 1
    elif day < 1:
        day = 30 + day #day is negative, so (+) to avoid positive from double negative
        month -= 1

    if month > 12:
        month = 1
        year += 1
    elif month < 1:
        month = 12
        year -= 1

    return f'{year:04d}{month:02d}{day:02d}'

In [4]:
# Check overlaps between drivers themselves
def test_coincidence(test_date_tuple, date_dict):
    ''' test_date_tuple is a tuple of (start_date, end_date) for the compound event
        date_dict is a dictionary where values are tuples of (start_date, end_date) for driver events
    '''
    
    yyyymmdd1, yyyymmdd2 = test_date_tuple
    result = list()
    for dd in date_dict.values():
        yyyymmdd3, yyyymmdd4 = dd
        overlap = not (yyyymmdd2 < yyyymmdd3 or yyyymmdd4 < yyyymmdd1)
        result.append(overlap)
    # print(result)
    return any(result)

In [ ]:
# Get the start date of the compound event, and the start dates of the driver events
def get_start_date(event):
    if event:
        return event[0][0]

def get_both_start_dates(event):
    if event:
        return event[0]

In [6]:
# Calculate t1, t2, t3 for each combination
def calculate_decomposition_terms(hist_df, fut_df, prefix):
    ''' calc decomposition terms for a given event type
    hist_df and fut_df should have columns:
    'ensemble' | 'p(C)' | 'p(C|A)' | 'p(C|B)' | 'p(C|S)' | 'p(A)' | 'p(B)' | 'p(S)'    

    '''
    # map prefix to event letter
    event_letter = prefix.split('_')[0][0]  # 'AR_only' -> 'A', 'Block_only' -> 'B', 'Storm_only' -> 'S'
    # use delta_p_C column which exists in the dataframes
    p_C_fut = fut_df['p(C)']
    p_C_hist = hist_df['p(C)']
    
    gamma_fut = fut_df[f'p(C|{event_letter})'] - p_C_fut
    gamma_hist = hist_df[f'p(C|{event_letter})'] - p_C_hist
    delta_gamma = gamma_fut - gamma_hist
    
    delta_p_event = fut_df[f'p({event_letter})'] - hist_df[f'p({event_letter})']
    delta_p_C = p_C_fut - p_C_hist
    
    t1 = delta_gamma * hist_df[f'p({event_letter})']
    t2 = delta_p_event * gamma_hist
    t3 = delta_gamma * delta_p_event
    
    return pd.DataFrame({
        'ensemble': hist_df['ensemble'],
        f'{prefix}_delta_p_event': delta_p_event,
        f'{prefix}_delta_p_C': delta_p_C,
        f'{prefix}_t_total': t1 + t2 + t3})

def calculate_decomposition_terms_multi(hist_df, fut_df, prefix, event_letters):
    # use delta_p_C from one of the columns (they're all the same)
    p_C_fut = fut_df['p(C)']  # p(C) is same for all
    p_C_hist = hist_df['p(C)']
    
    gamma_fut = fut_df[f'p(C|{event_letters})'] - p_C_fut
    gamma_hist = hist_df[f'p(C|{event_letters})'] - p_C_hist
    delta_gamma = gamma_fut - gamma_hist
    
    delta_p_event = fut_df[f'p({event_letters})'] - hist_df[f'p({event_letters})']
    delta_p_C = p_C_fut - p_C_hist
    
    t1 = delta_gamma * hist_df[f'p({event_letters})']
    t2 = delta_p_event * gamma_hist
    t3 = delta_gamma * delta_p_event
    
    return pd.DataFrame({
        'ensemble': hist_df['ensemble'],
        f'{prefix}_delta_p_event': delta_p_event,
        f'{prefix}_delta_p_C': delta_p_C,
        f'{prefix}_t_total': t1 + t2 + t3})


In [16]:
#function to check if a frver overlaps with compound event windw
def driver_overlaps(compound_start, compound_end, driver_start, driver_end):
    '''check if driver event overlaps with compound event window'''
    return (int(compound_start) <= int(driver_end)) and (int(compound_end) >= int(driver_start))

# causal framework, with three drivers and a compound events. 


It is entirely possible to add as many drivers as you want to this code, just requires some tweeking. 

In [14]:
period_start_hist, period_end_hist = 1980, 1999     #start and end year for historical period
period_start_fut, period_end_fut = 2060, 2079       #start and end year for future period
tau = -1                                            #lag in window of time, (-1) means we look at the day before the start of the compound event     

ensembles = ['2335','1649','0000','1554','1843','1935','2123','2242','2305','2491','2868','1113']


In [ ]:
# fc to check if date is in JJA months

all_combinations_results_hist = []
all_combinations_results_fut = []

for ensemble in ensembles:
    ar_events_df = pd.read_csv('/home/users/elena.dauster/Documents/python/CoinCalc/ARs/AR_events_start_end.csv')  #format: 'ensemble' | 'AR_start_date' | 'AR_end_date' | 'AR_duration'
    storms_events_df = pd.read_csv(f'/home/users/elena.dauster/Documents/python/CoinCalc/storms/storms_{ensemble}_grouped.csv')     #format: event_start | event_end | duration
    blocking_hist_df = pd.read_csv(f'/home/users/elena.dauster/Documents/python/CoinCalc/blocks/blocking_events_Iceland_{ensemble}_1979_18_grouped.csv')    #format: event_start | event_end | duration
    blocking_fut_df = pd.read_csv(f'/home/users/elena.dauster/Documents/python/CoinCalc/blocks/blocking_events_Iceland_{ensemble}_2060_79_grouped.csv')
    compound_start_date = pd.read_csv(f'/home/users/elena.dauster/Documents/python/CoinCalc/data/events_yyyymmdd/p110{ensemble}_events_20mmh_compound.csv') #format: start_date | length_days | extreme_days
    
    compound_jja = compound_start_date[compound_start_date['start_date'].astype(str).str[4:6].isin(['06','07','08'])]   #extracting only JJA events
    compound_filtered = compound_jja[compound_jja['extreme_days'] >= 2]     #can odify to selct longer events 
    compound_all = compound_filtered[['start_date', 'length_days']].dropna()

    #=======================================================================================================
    # ------- HISTORICAL PERIOD -------
    ar_hist = ar_events_df[
        (ar_events_df['ensemble'] == int(ensemble)) & 
        (ar_events_df['AR_start_date'].astype(str).str[:4].astype(int) >= period_start_hist) &
        (ar_events_df['AR_start_date'].astype(str).str[:4].astype(int) <= period_end_hist) &
        (ar_events_df['AR_duration'] >= 1)]
    ar_hist = ar_hist[ar_hist['AR_start_date'].apply(is_jja)]
    
    storms_hist = storms_events_df[
        (storms_events_df['event_start'].astype(str).str[:4].astype(int) >= period_start_hist) &
        (storms_events_df['event_start'].astype(str).str[:4].astype(int) <= period_end_hist) &
        (storms_events_df['duration'] >= 1)]
    storms_hist = storms_hist[storms_hist['event_start'].apply(is_jja)]
    
    #our blocking is jja only already, so no need to filter for jja months
    blocking_hist = blocking_hist_df[
        (blocking_hist_df['event_start'].astype(str).str[:4].astype(int) >= period_start_hist) &
        (blocking_hist_df['event_start'].astype(str).str[:4].astype(int) <= period_end_hist) &
        (blocking_hist_df['duration'] >= 1)]
    
    compound_hist = compound_all[
            (compound_all['start_date'].astype(str).str[:4].astype(int) >= period_start_hist) &
            (compound_all['start_date'].astype(str).str[:4].astype(int) <= period_end_hist)]

    compound_start_hist = compound_hist.apply(lambda row: adjust_date(str(row['start_date']), offset= tau ), axis=1)
    compound_end_hist = compound_hist.apply(lambda row: adjust_date(str(row['start_date']), offset=row['length_days']-1) , axis=1)  

    # check if compound data exists before creating DataFrame
    if len(compound_hist) > 0:
        compound_window_hist_df = pd.DataFrame({
            'start_date': compound_start_hist.values,  # Use .values instead of .reset_index(drop=True)
            'end_date': compound_end_hist.values})
    else:
        # CREATE empty DataFrame with correct columns if no compound events
        compound_window_hist_df = pd.DataFrame(columns=['start_date', 'end_date'])

    compound_window_hist_df = pd.DataFrame(compound_window_hist_df)

    num_years_hist = period_end_hist - period_start_hist + 1

    ###############################################################################################
    # ---------------------- HISTORICAL period -------------------------
    event_types = ['A', 'B', 'S']   #atomsperhic rives, blocks , storms
    ar_hist, blocking_hist, storms_hist

    ar_events = [(int(row['AR_start_date']), int(row['AR_end_date'])) for ir, row in ar_hist.iterrows()]
    bl_events = [(int(row['event_start']), int(row['event_end'])) for ir, row in blocking_hist.iterrows()]
    st_events = [(int(row['event_start']), int(row['event_end'])) for ir, row in storms_hist.iterrows()]

    all_events_dict = dict(zip(event_types, [ar_events, bl_events, st_events]))
    max_len_dict = dict([(et, len(all_events_dict[et])) for et in event_types])

    unique_sets = list()
    nevent = dict(zip(event_types, [0] * len(event_types)))

    #mapping for counting the combinations of drivers
    mapping = {(0,0,0):'none',
           (0,0,1):'S',
           (0,1,0):'B',
           (0,1,1):'B and S',
           (1,0,0):'A',
           (1,0,1):'A and S',
           (1,1,0):'A and B',
           (1,1,1):'A and B and S',
          }
    count_keys = ['A', 'B', 'S', 'A and B', 'A and S', 'B and S', 'A and B and S', 'none']
        
    while nevent != max_len_dict:
        #this loop goes through all events in chronological order, checking for overlaps and creating sets of coincident events.
        start_dates = dict([(et, get_start_date(all_events_dict[et][nevent[et]:])) for et in event_types if all_events_dict[et][nevent[et]:]])
        both_dates = dict([(et, get_both_start_dates(all_events_dict[et][nevent[et]:])) for et in event_types if all_events_dict[et][nevent[et]:]])
        # start_dates = dict(zip(event_types, [get_start_date(e) for e in all_events]))
        # both_dates = dict(zip(event_types, [get_both_start_dates(e) for e in all_events]))
        earliest_date = min(start_dates.values())
        include = dict([kv for kv in both_dates.items() if kv[1][0] == earliest_date])
        exclude = dict([kv for kv in both_dates.items() if kv[1][0] != earliest_date])

        possible_overlap = True
        # This loop checks if any of the excluded events overlap with the included events. If they do, they are added to the include set and removed from the exclude set. 
        # This continues until there are no more overlaps, ensuring that all coincident events are grouped together.
        while possible_overlap:
            sorted_excluded_keys = [k for k, v in sorted(exclude.items(), key=lambda item: item[1])]
            is_there_coincidence = [test_coincidence(exclude[kk], include) for kk in sorted_excluded_keys]
            if any(is_there_coincidence):
                key2add = sorted_excluded_keys[is_there_coincidence.index(True)]
                include[key2add] = exclude[key2add]
                del exclude[key2add]
            else:    # no coincidences, tidy up
                possible_overlap = False
                    
        unique_sets.append(include)
        for etype in include.keys():
            nevent[etype] += 1
    driver_count_dict_hist = dict(zip(count_keys, [0] * len(count_keys)))
    for u in unique_sets:
        # print(u)
        key = ('A' in u, 'B' in u, 'S' in u)
        driver_count_dict_hist[mapping[key]] += 1

    ########################
    # Now check compound events against driver overlaps
    combination_counts_hist = {
    'A_only': 0,
    'B_only': 0,
    'S_only': 0,
    'A_B': 0,
    'A_S': 0,
    'B_S': 0,
    'A_B_S': 0,
    'None': 0  }

    ############ check compound and driver overap
    # This loop goes through each compound event and checks if it overlaps with any of the driver events (ARs, blocks, storms). 
    # It then categorises the compound event based on which drivers are present and counts the combinations.
    for _, compound_event in compound_window_hist_df.iterrows():
        comp_start = compound_event['start_date']
        comp_end = compound_event['end_date']

        has_ar = False
        has_block = False
        has_storm = False

        # cehck aRs 
        for _, ar_event in ar_hist.iterrows():
            if driver_overlaps(comp_start, comp_end, ar_event['AR_start_date'], ar_event['AR_end_date']):
                has_ar = True
                break
            
        for _, block_event in blocking_hist.iterrows():
            if driver_overlaps(comp_start, comp_end, block_event['event_start'], block_event['event_end']):
                has_block = True
                break
            
        for _, storm_event in storms_hist.iterrows():
            if driver_overlaps(comp_start, comp_end, storm_event['event_start'], storm_event['event_end']):
                has_storm = True
                break
            
        # cat based on which drivers are present
        if has_ar and has_block and has_storm:
            combination_counts_hist['A_B_S'] += 1
        elif has_ar and has_block:
            combination_counts_hist['A_B'] += 1
        elif has_ar and has_storm:
            combination_counts_hist['A_S'] += 1
        elif has_block and has_storm:
            combination_counts_hist['B_S'] += 1
        elif has_ar:
            combination_counts_hist['A_only'] += 1
        elif has_block:
            combination_counts_hist['B_only'] += 1
        elif has_storm:
            combination_counts_hist['S_only'] += 1
        else:
            combination_counts_hist['None'] += 1

    num_compound_hist = len(compound_hist)
    total_days_hist = 90 * num_years_hist

    hist_probs = {
        'ensemble': ensemble,
        'p(C)' : num_compound_hist / total_days_hist,

        #AR only
        'p(A)' : driver_count_dict_hist['A'] / total_days_hist,
        'p(A and C)' : combination_counts_hist['A_only'] / total_days_hist,
        'p(C|A)' : combination_counts_hist['A_only'] / driver_count_dict_hist['A'] if driver_count_dict_hist['A'] > 0 else 0,
        'AR only count' : combination_counts_hist['A_only'],

        # B only
        'p(B)' : driver_count_dict_hist['B'] / total_days_hist,
        'p(B and C)' : combination_counts_hist['B_only'] / total_days_hist,
        'p(C|B)' : combination_counts_hist['B_only'] / driver_count_dict_hist['B'] if driver_count_dict_hist['B'] > 0 else 0,
        'B only count' : combination_counts_hist['B_only'],

        # S only
        'p(S)' : driver_count_dict_hist['S'] / total_days_hist,
        'p(S and C)' : combination_counts_hist['S_only'] / total_days_hist,
        'p(C|S)' : combination_counts_hist['S_only'] / driver_count_dict_hist['S'] if driver_count_dict_hist['S'] > 0 else 0,
        'S only count' : combination_counts_hist['S_only'],   

        # A and B
        'p(A and B)': driver_count_dict_hist['A and B'] / total_days_hist,
        'p(A and B and C)' : combination_counts_hist['A_B'] / total_days_hist, 
        'p(C|A and B)': combination_counts_hist['A_B'] / driver_count_dict_hist['A and B'] if driver_count_dict_hist['A and B'] > 0 else 0,
        'A and B count': driver_count_dict_hist['A and B'],

        #A and S
        'p(A and S)' : driver_count_dict_hist['A and S'] / total_days_hist,
        'p(A and S and C)': combination_counts_hist['A_S'] / total_days_hist, 
        'p(C|A and S)': combination_counts_hist['A_S'] / driver_count_dict_hist['A and S'] if driver_count_dict_hist['A and S'] > 0 else 0,
        'A and S count': driver_count_dict_hist['A and S'],

        #B and S
        'p(B and S)' : driver_count_dict_hist['B and S'] / total_days_hist,
        'p(B and S and C)': combination_counts_hist['B_S'] / total_days_hist, 
        'p(C|B and S)': combination_counts_hist['B_S'] / driver_count_dict_hist['B and S'] if driver_count_dict_hist['B and S'] > 0 else 0,
        'B and S count': driver_count_dict_hist['B and S'],

        # A and B and S
        'p(A and B and S)': driver_count_dict_hist['A and B and S'] / total_days_hist,
        'p(A and B and S and C)': combination_counts_hist['A_B_S'] / total_days_hist, 
        'p(C|A and B and S)' : combination_counts_hist['A_B_S'] / driver_count_dict_hist['A and B and S'] if driver_count_dict_hist['A and B and S'] > 0 else 0}

    all_combinations_results_hist.append(hist_probs)

    ###############################################################################
    ar_fut = ar_events_df[
        (ar_events_df['ensemble'] == int(ensemble)) & 
        (ar_events_df['AR_start_date'].astype(str).str[:4].astype(int) >= period_start_fut) &
        (ar_events_df['AR_start_date'].astype(str).str[:4].astype(int) <= period_end_fut) &
        (ar_events_df['AR_duration'] >= 1)]
    ar_fut = ar_fut[ar_fut['AR_start_date'].apply(is_jja)]
    
    storms_fut = storms_events_df[
        (storms_events_df['event_start'].astype(str).str[:4].astype(int) >= period_start_fut) &
        (storms_events_df['event_start'].astype(str).str[:4].astype(int) <= period_end_fut) &
        (storms_events_df['duration'] >= 1)]
    storms_fut = storms_fut[storms_fut['event_start'].apply(is_jja)]
    
    blocking_fut = blocking_fut_df[
        (blocking_fut_df['event_start'].astype(str).str[:4].astype(int) >= period_start_fut) &
        (blocking_fut_df['event_start'].astype(str).str[:4].astype(int) <= period_end_fut) &
        (blocking_fut_df['duration'] >= 1)]
    
    compound_fut = compound_all[
        (compound_all['start_date'].astype(str).str[:4].astype(int) >= period_start_fut) &
        (compound_all['start_date'].astype(str).str[:4].astype(int) <= period_end_fut)]
        

    compound_start_fut = compound_fut.apply(lambda row: adjust_date(str(row['start_date']), offset= tau ), axis=1)
    compound_end_fut = compound_fut.apply(lambda row: adjust_date(str(row['start_date']), offset=row['length_days']-1), axis=1)  

    compound_window_fut = {
        'start_date': compound_start_fut, 
        'end_date': compound_end_fut}
    
    compound_window_fut_df = pd.DataFrame(compound_window_fut)

    num_years_fut = period_end_fut - period_start_fut + 1
    # ------------------- future period  ------------------------
    event_types = ['A', 'B', 'S']
    ar_fut, blocking_fut, storms_fut

    ar_events = [(int(row['AR_start_date']), int(row['AR_end_date'])) for ir, row in ar_fut.iterrows()]
    bl_events = [(int(row['event_start']), int(row['event_end'])) for ir, row in blocking_fut.iterrows()]
    st_events = [(int(row['event_start']), int(row['event_end'])) for ir, row in storms_fut.iterrows()]
    all_events_dict = dict(zip(event_types, [ar_events, bl_events, st_events]))
    max_len_dict = dict([(et, len(all_events_dict[et])) for et in event_types])

    unique_sets = list()
    nevent = dict(zip(event_types, [0] * len(event_types)))


    mapping = {(0,0,0):'none',
           (0,0,1):'S',
           (0,1,0):'B',
           (0,1,1):'B and S',
           (1,0,0):'A',
           (1,0,1):'A and S',
           (1,1,0):'A and B',
           (1,1,1):'A and B and S',}

    count_keys = ['A', 'B', 'S', 'A and B', 'A and S', 'B and S', 'A and B and S', 'none']
        
    while nevent != max_len_dict:

        start_dates = dict([(et, get_start_date(all_events_dict[et][nevent[et]:])) for et in event_types if all_events_dict[et][nevent[et]:]])
        both_dates = dict([(et, get_both_start_dates(all_events_dict[et][nevent[et]:])) for et in event_types if all_events_dict[et][nevent[et]:]])
        # start_dates = dict(zip(event_types, [get_start_date(e) for e in all_events]))
        # both_dates = dict(zip(event_types, [get_both_start_dates(e) for e in all_events]))
        earliest_date = min(start_dates.values())
        include = dict([kv for kv in both_dates.items() if kv[1][0] == earliest_date])
        exclude = dict([kv for kv in both_dates.items() if kv[1][0] != earliest_date])

        possible_overlap = True
        while possible_overlap:
            sorted_excluded_keys = [k for k, v in sorted(exclude.items(), key=lambda item: item[1])]
            is_there_coincidence = [test_coincidence(exclude[kk], include) for kk in sorted_excluded_keys]
            if any(is_there_coincidence):
                key2add = sorted_excluded_keys[is_there_coincidence.index(True)]
                include[key2add] = exclude[key2add]
                del exclude[key2add]
            else:    # no coincidences, tidy up
                possible_overlap = False
                    
        unique_sets.append(include)
        for etype in include.keys():
            nevent[etype] += 1

    driver_count_dict_fut = dict(zip(count_keys, [0] * len(count_keys)))
    for u in unique_sets:
        # print(u)
        key = ('A' in u, 'B' in u, 'S' in u)
        driver_count_dict_fut[mapping[key]] += 1


##########################
    combination_counts_fut = {
    'A_only': 0,
    'B_only': 0,
    'S_only': 0,
    'A_B': 0,
    'A_S': 0,
    'B_S': 0,
    'A_B_S': 0,
    'None': 0 }

    for _, compound_event in compound_window_fut_df.iterrows():
        comp_start = compound_event['start_date']
        comp_end = compound_event['end_date']

        has_ar = False
        has_block = False
        has_storm = False

        # cehck aRs 
        for _, ar_event in ar_fut.iterrows():
            if driver_overlaps(comp_start, comp_end, ar_event['AR_start_date'], ar_event['AR_end_date']):
                has_ar = True
                break
            
        for _, block_event in blocking_fut.iterrows():
            if driver_overlaps(comp_start, comp_end, block_event['event_start'], block_event['event_end']):
                has_block = True
                break
            
        for _, storm_event in storms_fut.iterrows():
            if driver_overlaps(comp_start, comp_end, storm_event['event_start'], storm_event['event_end']):
                has_storm = True
                break
            
        # cat based on which drivers are present
        if has_ar and has_block and has_storm:
            combination_counts_fut['A_B_S'] += 1
        elif has_ar and has_block:
            combination_counts_fut['A_B'] += 1
        elif has_ar and has_storm:
            combination_counts_fut['A_S'] += 1
        elif has_block and has_storm:
            combination_counts_fut['B_S'] += 1
        elif has_ar:
            combination_counts_fut['A_only'] += 1
        elif has_block:
            combination_counts_fut['B_only'] += 1
        elif has_storm:
            combination_counts_fut['S_only'] += 1
        else:
            combination_counts_fut['None'] += 1

    num_compound_fut = len(compound_fut)
    total_days_fut = 90 * num_years_fut

    fut_probs = {
        'ensemble': ensemble,
        'p(C)' : num_compound_fut / total_days_fut,

        #AR only
        'p(A)' : driver_count_dict_fut['A'] / total_days_fut,
        'p(A and C)' : combination_counts_fut['A_only'] / total_days_fut,
        'p(C|A)' : combination_counts_fut['A_only'] / driver_count_dict_fut['A'] if driver_count_dict_fut['A'] > 0 else 0,
        'AR only count' : combination_counts_fut['A_only'],

        # B only
        'p(B)' : driver_count_dict_fut['B'] / total_days_fut,
        'p(B and C)' : combination_counts_fut['B_only'] / total_days_fut,
        'p(C|B)' : combination_counts_fut['B_only'] / driver_count_dict_fut['B'] if driver_count_dict_fut['B'] > 0 else 0,
        'B only count' : combination_counts_fut['B_only'],

        # S only
        'p(S)' : driver_count_dict_fut['S'] / total_days_fut,
        'p(S and C)' : combination_counts_fut['S_only'] / total_days_fut,
        'p(C|S)' : combination_counts_fut['S_only'] / driver_count_dict_fut['S'] if driver_count_dict_fut['S'] > 0 else 0,
        'S only count' : combination_counts_fut['S_only'],   

        # A and B
        'p(A and B)': driver_count_dict_fut['A and B'] / total_days_fut,
        'p(A and B and C)' : combination_counts_fut['A_B'] / total_days_fut, 
        'p(C|A and B)': combination_counts_fut['A_B'] / driver_count_dict_fut['A and B'] if driver_count_dict_fut['A and B'] > 0 else 0,
        'A and B count': driver_count_dict_fut['A and B'],

        #A and S
        'p(A and S)' : driver_count_dict_fut['A and S'] / total_days_fut,
        'p(A and S and C)': combination_counts_fut['A_S'] / total_days_fut, 
        'p(C|A and S)': combination_counts_fut['A_S'] / driver_count_dict_fut['A and S'] if driver_count_dict_fut['A and S'] > 0 else 0,
        'A and S count': driver_count_dict_fut['A and S'],

        #B and S
        'p(B and S)' : driver_count_dict_fut['B and S'] / total_days_fut,
        'p(B and S and C)': combination_counts_fut['B_S'] / total_days_fut, 
        'p(C|B and S)': combination_counts_fut['B_S'] / driver_count_dict_fut['B and S'] if driver_count_dict_fut['B and S'] > 0 else 0,
        'B and S count': driver_count_dict_fut['B and S'],

        # A and B and S
        'p(A and B and S)': driver_count_dict_fut['A and B and S'] / total_days_fut,
        'p(A and B and S and C)': combination_counts_fut['A_B_S'] / total_days_fut, 
        'p(C|A and B and S)' : combination_counts_fut['A_B_S'] / driver_count_dict_fut['A and B and S'] if driver_count_dict_fut['A and B and S'] > 0 else 0,
        'A and B and S count': driver_count_dict_fut['A and B and S']}

    all_combinations_results_fut.append(fut_probs)

###########################################################################################################
combinations_hist_df = pd.DataFrame(all_combinations_results_hist)
combinations_fut_df = pd.DataFrame(all_combinations_results_fut)

# Calculate terms for each combination
ar_only_terms = calculate_decomposition_terms(combinations_hist_df, combinations_fut_df, 'A_only')
block_only_terms = calculate_decomposition_terms(combinations_hist_df, combinations_fut_df, 'B_only')
storm_only_terms = calculate_decomposition_terms(combinations_hist_df, combinations_fut_df, 'S_only')

ar_block_terms = calculate_decomposition_terms_multi(combinations_hist_df, combinations_fut_df, 'A_B', 'A and B')
ar_storm_terms = calculate_decomposition_terms_multi(combinations_hist_df, combinations_fut_df, 'A_S', 'A and S')
block_storm_terms = calculate_decomposition_terms_multi(combinations_hist_df, combinations_fut_df, 'B_S', 'B and S')
ar_block_storm_terms = calculate_decomposition_terms_multi(combinations_hist_df, combinations_fut_df, 'A_B_S', 'A and B and S')

all_decomposition_results = pd.concat([
    combinations_hist_df[['ensemble']],
    ar_only_terms.drop('ensemble', axis=1),
    block_only_terms.drop('ensemble', axis=1),
    storm_only_terms.drop('ensemble', axis=1),
    ar_block_terms.drop('ensemble', axis=1),
    ar_storm_terms.drop('ensemble', axis=1),
    block_storm_terms.drop('ensemble', axis=1),
    ar_block_storm_terms.drop('ensemble', axis=1)
], axis=1)

all_decomposition_results = all_decomposition_results.sort_values('A_only_delta_p_C', ascending=True).reset_index(drop=True)

In [17]:
print(all_decomposition_results)

   ensemble  A_only_delta_p_event  A_only_delta_p_C  A_only_t_total  \
0      2335              0.022778          0.003333       -0.000328   
1      2491              0.018889          0.007778        0.000310   
2      1649              0.026667          0.010556       -0.001723   
3      1843              0.015000          0.011111        0.000785   
4      2868              0.012778          0.012778        0.000545   
5      1935              0.008889          0.013889        0.000638   
6      2123              0.040556          0.013889        0.000255   
7      0000              0.001111          0.018333       -0.001127   
8      2242              0.023889          0.021111       -0.000439   
9      1113              0.021667          0.022778        0.000526   
10     2305             -0.002222          0.025000        0.000732   
11     1554              0.007222          0.025556        0.002547   

    B_only_delta_p_event  B_only_delta_p_C  B_only_t_total  \
0             